# Pattern 2: Model Registry sync + `ModelBuilder` repack

In this pattern the **SageMaker AI Model Registry** becomes the governance hub. The
MLflow app's automatic model registration syncs every `mlflow.register_model()` call
into a Model Package — but that package is *metadata-only*: it has no deployable
artifacts. **`ModelBuilder`** closes the gap: we download the model from MLflow **at
build time** and `ModelBuilder` repacks it together with **auto-generated inference
code** (`inference.py`, request/response handlers) into a deployable S3 archive,
which we then attach to the synced Model Package.

```
MLflow ──register──▶ auto-sync ──▶ Model Package (metadata only)
   │                                        ▲
   └─download──▶ ModelBuilder.build() ──────┘ update_model_package
     (build      repacked archive in S3       (InferenceSpecification)
      time)      = model copy + generated
                   inference.py
```

Because the model bytes are baked into the archive, the endpoint **never calls the
MLflow app at runtime** — instance launches and scale-out events depend only on S3.

**When to choose this pattern:**
- you need the SageMaker Model Registry for governance (approvals, lineage,
  cross-account sharing) — and
- you don't want to write the serving handler by hand: you provide only a small
  `load()`/`invoke()` class and `ModelBuilder` generates the rest

**Trade-offs:**
- the deployable artifact is a **copy** of the model, repacked outside MLflow — the
  MLflow artifact store is no longer the single source of the served bytes
- an extra build step in your registration workflow (locally or in a pipeline step)


In [ ]:
%store -r mlflow_app_arn
%store -r mlflow_run_id
%store -r mlflow_model_ids
%store -r model_base_name
%store -r execution_role
%store -r region

import json
import time

import boto3
import mlflow

mlflow.set_tracking_uri(mlflow_app_arn)
sm_client = boto3.client("sagemaker")

## Step 1: Register the model — auto-sync creates the Model Package

Registration triggers the sync: a Model Package Group (with a short hash suffix on
the name) and a Model Package version are created automatically, carrying the MLflow
metrics, evaluation results, and lineage.

In [ ]:
registered_name = f"{model_base_name}-modelbuilder"

mv = mlflow.register_model(f"models:/{mlflow_model_ids['modelbuilder']}", registered_name)
model_uri = f"models:/{registered_name}/{mv.version}"
print(f"Registered {registered_name} v{mv.version}")

### Find the auto-created Model Package

The sync writes the new Model Package ARN back onto the MLflow model version as a
`sagemaker.model_package_arn` tag. Registration returns before the sync finishes,
so we poll until the tag appears. (The package's `MetadataProperties.GeneratedBy`
also records the MLflow registered-model name and version — useful if you ever
need to match from the SageMaker side instead.)

In [ ]:
# The sync runs asynchronously: poll the model version until the tag appears.
mlflow_client = mlflow.MlflowClient()

sm_model_package_arn = None
for _ in range(20):
    mv_get = mlflow_client.get_model_version(registered_name, mv.version)
    sm_model_package_arn = mv_get.tags.get("sagemaker.model_package_arn")
    if sm_model_package_arn:
        break
    print(".", end="", flush=True)
    time.sleep(3)

if not sm_model_package_arn:
    raise TimeoutError("SageMaker Model Package was not auto-created within timeout")
print(f"\nSynced Model Package: {sm_model_package_arn}")

In SageMaker Studio (**Models → Registered models**), the synced package carries the
training and evaluation results from MLflow — but its **Containers** section is empty:
nothing to deploy yet.

![The synced Model Package: Train/Evaluate Complete, but Containers and Instances are empty](img/metadata-sync-no-inference.png)

## Step 2: Repack the MLflow model with `ModelBuilder`

Two things happen here:

1. **Download the model at build time.** `mlflow.artifacts.download_artifacts()`
   pulls the full MLflow model directory (`MLmodel`, `model.pkl`, environment files)
   to local disk. Passing that directory to `ModelBuilder` as `model_path` puts the
   model bytes **inside** the repacked archive.
2. **Describe how to serve it.** The **inference specification** — a small class
   answering two questions you already know how to answer: *how do I load my model?*
   and *how do I predict with it?* — plus a `SchemaBuilder` with a sample input and
   output. From those, `ModelBuilder` generates the serving code (`inference.py`,
   request/response handling) and packages everything into a deployable archive in S3.

`load()` reads the packaged copy from the container-local `model_dir` with
`mlflow.pyfunc` — the `MLmodel` metadata travels with the artifacts, so the same
class generalizes to other MLflow flavors (though the serving image, dependencies,
and schema may need to change with the framework).

> **Why not load from MLflow at runtime?** The spec *could* call
> `mlflow.pyfunc.load_model("models:/...")` inside the container instead. That works,
> but it couples every instance launch — including auto-scaling events — to the
> availability of (and network access + credentials for) the MLflow app. Downloading
> at build time keeps endpoint startup deterministic and S3-only.

> The SageMaker docs also describe a `model_metadata={"MLFLOW_MODEL_PATH": ...}`
> shortcut. We use the explicit `InferenceSpec` here because it gives full control
> over loading and prediction.


In [ ]:
from pathlib import Path

import numpy as np
from sagemaker.core import image_uris
from sagemaker.serve import ModelBuilder
from sagemaker.serve.spec.inference_spec import InferenceSpec
from sagemaker.serve.builder.schema_builder import SchemaBuilder
from sagemaker.serve.utils.types import ModelServer

# Download the MLflow model directory (MLmodel, model.pkl, env files) at build
# time. ModelBuilder packs these bytes into the repacked archive, so the endpoint
# never needs to reach the MLflow app.
local_model_dir = mlflow.artifacts.download_artifacts(
    artifact_uri=model_uri, dst_path="mlflow_model_download"
)
print(f"Downloaded model to: {local_model_dir}")
print(f"Contents: {sorted(p.name for p in Path(local_model_dir).iterdir())}")


class MLflowPyfuncSpec(InferenceSpec):
    """How to load the packaged MLflow model and how to predict with it."""

    def load(self, model_dir):
        import mlflow as _mlflow
        # model_dir is /opt/ml/model inside the container: the repacked copy of
        # the MLflow model directory. Local load — no tracking-server call.
        return _mlflow.pyfunc.load_model(model_dir)

    def invoke(self, input_object, model):
        import numpy as np
        return model.predict(np.array(input_object)).tolist()


# Always pin the serving container explicitly — resolved for the current
# region with the SDK v3 image_uris helper, no hardcoded ECR URIs.
sklearn_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.4-2-py312",
    image_scope="inference",
    instance_type="ml.m5.xlarge",
)
print(f"Serving container image: {sklearn_image}")

model_builder = ModelBuilder(
    image_uri=sklearn_image,
    model_path=local_model_dir,          # the downloaded MLflow model directory
    inference_spec=MLflowPyfuncSpec(),
    schema_builder=SchemaBuilder(
        sample_input=[[0.5, -1.2, 0.3, 0.8]],
        sample_output=[42.0],
    ),
    model_server=ModelServer.TORCHSERVE,
    role_arn=execution_role,
    instance_type="ml.m5.xlarge",
    # load() runs inside the serving container, so mlflow must be installed there
    # to read the packaged MLmodel; the generated inference.py imports the
    # sagemaker SDK.
    dependencies={"auto": False, "custom": [
        "mlflow<4",
        "sagemaker>=3,<4",
        "scikit-learn>=1.4,<1.5",
        # The sklearn container's serving framework (sagemaker-containers) ships
        # protobuf-generated code that requires protobuf<3.21. Without this pin,
        # mlflow's transitive deps upgrade protobuf and the serve process crashes
        # with 'Descriptors cannot be created directly'.
        "protobuf<3.21",
        # The container serves with Flask 1.1.1, which does `from jinja2 import
        # escape`. sagemaker-train (pulled in by sagemaker>=3) requires jinja2>=3.0,
        # and jinja2 3.1 removed `escape`. Jinja2 3.0.x satisfies both.
        "jinja2>=3.0,<3.1",
        "markupsafe<3",
        # setuptools 71+ puts its _vendor dir on the import path; the vendored (old)
        # typing_extensions shadows the real one, breaking pydantic_core
        # ("cannot import name 'Sentinel'"). setuptools 81+ also removes
        # pkg_resources, which sagemaker-containers imports. <70 avoids both.
        "setuptools<70",
    ]},
)

built_model = model_builder.build()

model_data_url = model_builder.s3_upload_path   # the repacked archive
serve_image_uri = sklearn_image
secret_key = model_builder.secret_key           # signs the generated code

print(f"Repacked model data: {model_data_url}")
print(f"Serving image:       {serve_image_uri}")

## Step 3: Attach the deployable artifacts to the synced Model Package

`update_model_package` fills the `InferenceSpecification` on the *auto-created*
package, so there is still **one** registry entry per model version — now a
deployable one.

> Calling `model_builder.register()` instead would create a *second*, separate Model
> Package — losing the link to the MLflow-synced entry and its lineage. Updating the
> synced package keeps the registries consistent.

In [ ]:
environment = {
    "SAGEMAKER_PROGRAM": "inference.py",
    "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
}
if secret_key:
    # ModelBuilder signs its generated artifacts; the endpoint verifies with this key.
    environment["SAGEMAKER_SERVE_SECRET_KEY"] = secret_key

sm_client.update_model_package(
    ModelPackageArn=sm_model_package_arn,
    InferenceSpecification={
        "Containers": [{
            "Image": serve_image_uri,
            "ModelDataUrl": model_data_url,
            "Environment": environment,
        }],
        "SupportedContentTypes": ["application/json"],
        "SupportedResponseMIMETypes": ["application/json"],
        "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.xlarge"],
    },
    CustomerMetadataProperties={
        "MlflowModelUri": model_uri,
        "MlflowRunId": mlflow_run_id,
    },
)
print("Model Package updated with deployable InferenceSpecification.")

The same package in Studio now shows the serving container, the repacked model
location in S3, and the two environment variables — it is deployable:

![The Model Package after update_model_package: Containers section filled with image URI, model data URL, and environment variables](img/model-builder-inference-metadata.png)

## Step 4: Approve and deploy from the Model Registry

From here on, deployment is registry-driven — identical to pattern 3 and to what the
**Deploy** button in Studio does. We use the SDK v3 **typed resources**.

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig, Model, ModelPackage
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

model_package = ModelPackage.get(model_package_name=sm_model_package_arn)
# Workaround: DescribeModelPackage does not return ModelPackageName for versioned
# packages, which breaks refresh()/update(). The API accepts an ARN in that field,
# so backfill it.
model_package.model_package_name = model_package.model_package_arn

model_package.update(model_approval_status="Approved")
print("Model Package approved.")

The Deploy stage flips to **Approved** — the promotion signal a CI/CD pipeline
(or the Studio **Deploy** button) reacts to:

![The Model Package showing Approved status with the Deploy button active](img/model-builder-inference-metadata-approved.png)

In [ ]:
from mlflow.models import get_model_info

# Check the model signature before deploying to SageMaker: without a signature
# the serving container cannot validate inference payloads.
model_info = get_model_info(model_uri)
if model_info.signature is None:
    raise ValueError(f"Model {model_uri} has no signature — refusing to deploy.")
print(f"Model signature OK: {model_info.signature}")

suffix = time.strftime("%Y%m%d-%H%M%S")
resource_name = f"{model_base_name}-mb-{suffix}"

deployed_model = Model.create(
    model_name=resource_name,
    primary_container=ContainerDefinition(model_package_name=sm_model_package_arn),
    execution_role_arn=execution_role,
)
endpoint_config = EndpointConfig.create(
    endpoint_config_name=resource_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=resource_name,
            initial_instance_count=1,
            instance_type="ml.m5.xlarge",
            # Server-side: fail the deployment if the container isn't healthy
            # or the model artifacts aren't downloaded within 7.5 minutes.
            container_startup_health_check_timeout_in_seconds=450,
            model_data_download_timeout_in_seconds=450,
        )
    ],
)
endpoint = Endpoint.create(
    endpoint_name=resource_name,
    endpoint_config_name=resource_name,
)
print(f"Creating endpoint {resource_name} (takes a few minutes)...")

endpoint_name = endpoint.endpoint_name

In [ ]:
print(f"Polling endpoint: {endpoint_name}")

terminal_states = {"InService", "Failed"}
while True:
    desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print(f"  {time.strftime('%H:%M:%S')} | status={status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != "InService":
    raise RuntimeError(
        f"Endpoint {endpoint_name} deployment ended with status '{status}': "
        f"{desc.get('FailureReason')}"
    )
print(f"Endpoint {endpoint_name} is InService.")

# Re-obtain the Endpoint resource for downstream .invoke() calls
endpoint = Endpoint.get(endpoint_name=endpoint_name)

In [ ]:
response = endpoint.invoke(
    body=json.dumps([[0.5, -1.2, 0.3, 0.8], [1.1, 0.4, -0.7, 0.2]]),
    content_type="application/json",
    accept="application/json",
)
print("Prediction:", response.body.read().decode())

## What you get — and what you don't

✅ Single governed registry entry per model version, now directly deployable
✅ **No hand-written serving handler or request-parsing boilerplate** — you write
   only `load()` and `invoke()`; `ModelBuilder` generates the rest
✅ No runtime dependency on the MLflow app: the endpoint serves from the repacked
   copy in S3, so scale-out never calls MLflow
✅ Works in a pipeline registration step

❌ The served artifact is a repacked **copy** outside the MLflow artifact store
❌ Extra build step (and its dependencies) in the registration path

Pattern 3 removes the repacking: the endpoint downloads the model **directly from
the MLflow artifact location**, with the inference specification logged through
`sagemaker-mlflow`.


## Teardown (and run `04_cleanup.ipynb` at the end)

The cells below use the resource objects created earlier in this notebook — if the
kernel restarted since deployment, skip them and run `04_cleanup.ipynb` instead.

In [ ]:
endpoint.delete()
endpoint_config.delete()
deployed_model.delete()